In [1]:
import cv2
import mediapipe as mp
import math

# Webcam
cap = cv2.VideoCapture(0)

# Face Mesh
mpFaceMesh = mp.solutions.face_mesh
faceMesh = mpFaceMesh.FaceMesh(
    max_num_faces=2,
    refine_landmarks=True
)

# Blink variables
blink_count = 0
blink_detected = False

while True:

    success, img = cap.read()

    if not success:
        break

    img = cv2.flip(img, 1)

    h, w, c = img.shape

    # Convert image
    imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Process face
    results = faceMesh.process(imgRGB)

    if results.multi_face_landmarks:

        for faceLms in results.multi_face_landmarks:

            # LEFT EYE
            left_top = faceLms.landmark[159]
            left_bottom = faceLms.landmark[145]

            # RIGHT EYE
            right_top = faceLms.landmark[386]
            right_bottom = faceLms.landmark[374]

            # Convert to pixels
            lx1, ly1 = int(left_top.x * w), int(left_top.y * h)
            lx2, ly2 = int(left_bottom.x * w), int(left_bottom.y * h)

            rx1, ry1 = int(right_top.x * w), int(right_top.y * h)
            rx2, ry2 = int(right_bottom.x * w), int(right_bottom.y * h)

            # Draw eye points
            cv2.circle(img, (lx1, ly1), 4, (0,255,0), -1)
            cv2.circle(img, (lx2, ly2), 4, (0,255,0), -1)

            cv2.circle(img, (rx1, ry1), 4, (0,255,0), -1)
            cv2.circle(img, (rx2, ry2), 4, (0,255,0), -1)

            # Eye distances
            left_eye_distance = math.hypot(lx2 - lx1, ly2 - ly1)
            right_eye_distance = math.hypot(rx2 - rx1, ry2 - ry1)

            # Average eye distance
            eye_distance = (
                left_eye_distance + right_eye_distance
            ) / 2

            # Blink detection
            if eye_distance < 6:

                if not blink_detected:
                    blink_count += 1
                    blink_detected = True

            else:
                blink_detected = False

    # Title
    cv2.putText(
        img,
        "AI BLINK GAME 👀",
        (10,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,255),
        3
    )

    # Blink count
    cv2.putText(
        img,
        f"Blinks: {blink_count}",
        (10,90),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255,255,0),
        3
    )

    # Winning message
    if blink_count >= 5:

        cv2.putText(
            img,
            "YOU WIN! 🎉",
            (120,220),
            cv2.FONT_HERSHEY_SIMPLEX,
            2,
            (0,255,0),
            5
        )

    cv2.imshow("AI Blink Game", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.r
    elease()
cv2.destroyAllWindows()